# LT-1 (1.00B) Google Colab L4 / A100 Training Notebook

Cost-optimal training pipeline for **LT-1 1.00B**:
- Uses **NVIDIA L4 (24 GB)** (~3.5-4.8 compute units/hr, lowest total cost per token) or **A100 (40 GB)**.
- Keeps the 469.8M Engram tables on GPU (`--profile colab`, zero PCIe overhead) with FP32 master weights in `Muon` + FP32 `AdamW` parameters.
- Builds a 150M-token 5-domain corpus (`FineWeb-Edu`, `Magicoder`, `MetaMathQA`, `Finance-Alpaca`, `UltraChat 200k`) + 50,000 causal SFT turns.
- Saves all checkpoints and exported inference packages directly to **Google Drive**.

In [ ]:
#@title 1. Check GPU & Mount Google Drive
import torch, os
from google.colab import drive

drive.mount('/content/drive')
DRIVE_CKPT_DIR = '/content/drive/MyDrive/lt1_colab_checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

assert torch.cuda.is_available(), 'CUDA GPU not found! Change Runtime -> Change runtime type -> L4 or A100.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'Active GPU: {gpu_name} ({vram_gb:.2f} GB VRAM) | BF16 Native: {torch.cuda.is_bf16_supported()}')
print(f'Checkpoints will persist in: {DRIVE_CKPT_DIR}')

In [ ]:
#@title 2. Clone LT-1 Repository & Install Dependencies
%cd /content
!if [ ! -d lt1 ]; then git clone https://github.com/LTgitspace/lt1.git; else git -C lt1 pull; fi
%cd /content/lt1
!pip -q install -U datasets tokenizers huggingface_hub
!python -m unittest discover -s tests -t . -v

In [ ]:
#@title 3. Build Scaled 5-Domain Pre-Training (150M Tokens) & SFT (50k Turns) Shards
#@markdown Streams data directly onto Colab NVMe disk (~2-4 minutes).
TARGET_TOKENS = 150000000 #@param {type:"integer"}
MAX_SFT_TURNS = 50000 #@param {type:"integer"}
SEQ_LEN = 512 #@param {type:"integer"}

!python -u -m src.data --mode scaled --tokenizer data/tokenizer_cosmo.json --out-bin data/train_mix_scaled.bin --out-sft data/sft_mix_scaled.pt --target-tokens {TARGET_TOKENS} --max-sft-turns {MAX_SFT_TURNS} --seq-len {SEQ_LEN}

In [ ]:
#@title 4. Run 1B Pre-Training + Auto-Chained Conversational SFT
#@markdown Automatically tunes Batch/GradAccum for L4 (24GB) vs A100 (40GB) and resumes from Google Drive if disconnected.
import os, torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
if vram_gb >= 36.0:
    BATCH_SIZE, GRAD_ACCUM = 32, 1  # A100 40GB: 16,384 tokens/step
else:
    BATCH_SIZE, GRAD_ACCUM = 16, 2  # L4 24GB: 16,384 tokens/step

STEPS = 12000       #@param {type:"integer"}
SFT_STEPS = 2000    #@param {type:"integer"}
MAX_MINUTES = 660   #@param {type:"integer"}
RESUME_FLAG = '--resume' if os.path.exists('/content/drive/MyDrive/lt1_colab_checkpoints/latest.pt') else ''

print(f'Launching: Batch={BATCH_SIZE}, Accum={GRAD_ACCUM}, SeqLen={SEQ_LEN}, EffectiveBatch={BATCH_SIZE*GRAD_ACCUM}, Resume={bool(RESUME_FLAG)}')
!python -u -m src.train \
  --stage pretrain \
  --scale 1b \
  --profile colab \
  --steps {STEPS} \
  --batch-size {BATCH_SIZE} \
  --seq-len {SEQ_LEN} \
  --grad-accum-steps {GRAD_ACCUM} \
  --loops 2 \
  --lr 4e-4 \
  --warmup-steps 200 \
  --save-interval 1000 \
  --eval-interval 250 \
  --max-minutes {MAX_MINUTES} \
  --data-bin data/train_mix_scaled.bin \
  --tokenizer-path data/tokenizer_cosmo.json \
  --checkpoint-dir /content/drive/MyDrive/lt1_colab_checkpoints \
  --log-file /content/drive/MyDrive/lt1_colab_checkpoints/train_colab.log \
  --chain-sft \
  --sft-pt data/sft_mix_scaled.pt \
  --sft-steps {SFT_STEPS} \
  {RESUME_FLAG}

In [ ]:
#@title 5. Quick Interactive Chat Smoke Test on Trained Checkpoint
from src.export import load_exported_checkpoint
from src.generate import generate_text

model, tok = load_exported_checkpoint('/content/drive/MyDrive/lt1_colab_checkpoints/lt1_1b_chat_inference.pt', device='cuda')
for prompt in [
    'User: What does profit margin mean in finance?\nAssistant:',
    'User: Write a Python function to compute Fibonacci numbers.\nAssistant:'
]:
    print('PROMPT:', prompt)
    print('REPLY:', generate_text(model, tok, prompt, max_new_tokens=80, num_loops=2, temperature=0.7))
    print('-' * 50)